# Как настроить BERT для классификации отзывов

На семинаре мы сравнили TF-IDF и замороженный BERT на отзывах. Теперь попробуем менять веса BERT, чтобы он лучше предсказывал класс рейтинга. Сначала повторим два знакомых способа, затем дообучим BERT и сравним все три на новых отзывах.

## 0. Данные и разбиение

In [ ]:
!pip install -q "transformers>=4.45,<5" "scikit-learn>=1.4,<2" pandas scipy

In [ ]:
import copy
import random

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F

from IPython.display import display
from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from tqdm.auto import tqdm
from transformers import AutoModel, AutoModelForSequenceClassification, AutoTokenizer, DataCollatorWithPadding, set_seed

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
set_seed(SEED)

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

print("Устройство:", DEVICE)

In [ ]:
WB_REVIEWS_URL = (
    "https://huggingface.co/datasets/Hplss/wb-review-dataset/"
    "resolve/main/filtered.csv"
)
LABEL_NAMES = np.array(["отрицательный", "нейтральный", "положительный"])

reviews = pd.read_csv(WB_REVIEWS_URL)
parts = reviews[["text", "pros", "cons"]].fillna("").astype(str)
reviews["input"] = parts.agg(" ".join, axis=1).str.strip()
reviews = reviews[reviews["input"].str.len() > 10].copy()
reviews["label"] = np.select(
    [reviews["rating"] <= 2, reviews["rating"] == 3], [0, 1], default=2
)
reviews = reviews.reset_index(drop=True)

display(reviews[["rating", "category_label", "input"]].head(3))
print("Всего отзывов:", len(reviews))

Отзывы об одном товаре часто похожи, поэтому оставим все отзывы с одним `product_id` в одной части данных. На train обучаем модели. На validation сравниваем промежуточные результаты и выбираем лучшую эпоху BERT. На test проверяем все три способа после выбора.

In [ ]:
def group_split(frame, test_size, seed):
    splitter = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=seed)
    left, right = next(splitter.split(frame, groups=frame["product_id"]))
    return frame.iloc[left].copy(), frame.iloc[right].copy()


train_val, test_full = group_split(reviews, test_size=0.2, seed=42)
train_full, val_full = group_split(train_val, test_size=0.25, seed=43)
train, _ = train_test_split(train_full, train_size=6000, stratify=train_full["label"], random_state=42)
val, _ = train_test_split(val_full, train_size=1500, stratify=val_full["label"], random_state=43)
test, _ = train_test_split(test_full, train_size=2000, stratify=test_full["label"], random_state=44)

assert set(train["product_id"]).isdisjoint(val["product_id"])
assert set(train["product_id"]).isdisjoint(test["product_id"])
assert set(val["product_id"]).isdisjoint(test["product_id"])
print("train:", len(train), "val:", len(val), "test:", len(test))

In [ ]:
balance = (
    train["label"].value_counts(normalize=True).sort_index()
    .rename(index=dict(enumerate(LABEL_NAMES)))
    .rename("доля в train")
)
display(balance.to_frame())

## 1. Первый вариант с TF-IDF

Сначала повторим знакомый способ. Признаки по словам помогают заметить фразы вроде «не подошёл», а признаки по символам могут связать опечатку с привычным написанием.

**Задача 1.** Постройте словные и символьные признаки для всех трёх частей данных. Оба словаря обучите на train. Преобразуйте validation и test с этими же словарями.

<details>
<summary><b>Нужна подсказка?</b></summary>

Нужны два независимых `TfidfVectorizer`: один для слов, второй с `analyzer="char_wb"`. Вспомните разницу между `fit_transform` и `transform` и не обучайте словари на validation или test.

</details>

In [ ]:
# YOUR CODE HERE  ฅ^•ﻌ•^ฅ
# Создайте word и char vectorizer.
# Получите x_train_word, x_val_word, x_test_word,
# x_train_char, x_val_char и x_test_char.

In [ ]:
assert x_train_word.shape[0] == len(train)
assert x_val_char.shape[0] == len(val)
assert x_test_char.shape[0] == len(test)
print(x_train_word.shape, x_train_char.shape)

**Задача 2.** Соедините два набора признаков и обучите на них одну логистическую регрессию. Оцените её на validation по accuracy и macro-F1.

<details>
<summary><b>Нужна подсказка?</b></summary>

Разреженные признаки объединяются через `hstack`. После этого используйте обычные `fit` и `predict` у `LogisticRegression`. Проверьте порядок строк во всех матрицах.

</details>

In [ ]:
# YOUR CODE HERE  ฅ^•ﻌ•^ฅ
# Объедините word и char признаки в x_train, x_val и x_test.
# Обучите tfidf_clf и получите pred_val_tfidf.

## 2. Замороженный BERT

Заменим TF-IDF на эмбеддинги BERT, как на семинаре. Сначала напишите `mean_pool`, чтобы получить один эмбеддинг на отзыв. При усреднении пропускайте `[PAD]`. Затем обучите логистическую регрессию на этих эмбеддингах. Веса BERT на этом шаге остаются прежними.

In [ ]:
BERT_ID = "cointegrated/rubert-tiny2"
bert_tokenizer = AutoTokenizer.from_pretrained(BERT_ID)
bert_encoder = AutoModel.from_pretrained(BERT_ID).to(DEVICE).eval()

<details>
<summary><b>Нужна подсказка?</b></summary>

Сначала сделайте форму attention mask совместимой с hidden states. Затем padding должен дать нулевой вклад и в сумму, и в число токенов. Пригодятся `unsqueeze`, `sum` и `clamp`.

</details>

In [ ]:
def mean_pool(hidden, attention_mask):
    # YOUR CODE HERE  ฅ^•ﻌ•^ฅ
    pass

In [ ]:
def embed_texts(texts, batch_size=None, max_length=128):
    if batch_size is None:
        batch_size = 128 if DEVICE.type == "cuda" else 32
    vectors = []
    for start in tqdm(range(0, len(texts), batch_size), leave=False):
        batch = bert_tokenizer(
            texts[start:start + batch_size], padding=True, truncation=True,
            max_length=max_length, return_tensors="pt",
        ).to(DEVICE)
        with torch.inference_mode():
            hidden = bert_encoder(**batch).last_hidden_state
        vectors.append(F.normalize(mean_pool(hidden, batch["attention_mask"]), dim=1).cpu())
    return torch.cat(vectors).numpy()


train_emb = embed_texts(train["input"].tolist())
val_emb = embed_texts(val["input"].tolist())
test_emb = embed_texts(test["input"].tolist())

<details>
<summary><b>Нужна подсказка?</b></summary>

BERT здесь уже ничего не обучает: `train_emb` и `val_emb` — готовые признаки. Нужны `LogisticRegression`, `accuracy_score` и `f1_score(..., average="macro")`.

</details>

In [ ]:
# YOUR CODE HERE  ฅ^•ﻌ•^ฅ
# Обучите frozen_clf на train_emb, получите pred_val_frozen и выведите две метрики.

## 3. Дообучаем BERT

Теперь будем обучать сам BERT. На каждом шаге он получает батч отзывов и правильные классы. Начнём с `ReviewDataset`, который выдаёт по одному примеру.

**Задача.** По индексу верните токены одного отзыва и его класс под ключом `labels`. Слишком длинный отзыв обрежьте до `max_length` токенов. До общей длины дополним тексты при сборке батча.

[Документация Dataset и DataLoader](https://pytorch.org/docs/stable/data.html)

<details>
<summary><b>Нужна подсказка?</b></summary>

`Dataset` должен уметь ответить на два вопроса: сколько в нём объектов и как получить один объект по индексу. Токенизируйте один текст без padding и добавьте его числовую метку под ключом `labels`.

</details>

In [ ]:
# YOUR CODE HERE  ฅ^•ﻌ•^ฅ
from torch.utils.data import DataLoader, Dataset

class ReviewDataset(Dataset):
    def __init__(self, frame, tokenizer, max_length=128):
        pass

    def __len__(self):
        pass

    def __getitem__(self, index):
        pass

In [ ]:
train_dataset = ReviewDataset(train, bert_tokenizer)
example = train_dataset[0]
assert "input_ids" in example and "labels" in example
print("Объектов:", len(train_dataset), "токенов в первом:", len(example["input_ids"]))

При сборке батча дополняем короткие отзывы до длины самого длинного отзыва в этом батче. Например, если в батче отзывы на 5 и 12 токенов, к первому добавим 7 токенов `[PAD]`. В другом батче максимум может быть другим. Поэтому такой padding называют динамическим.

`DataLoader` берёт из `Dataset` очередную группу примеров и передаёт её `collator`. Collator собирает из этих примеров батч тензоров. В нашем коде `DataCollatorWithPadding` вызывает `tokenizer.pad`, который добавляет токены `[PAD]`.

**Задача.** Соберите три `DataLoader` и передайте каждому готовый `collator`. Перемешивайте отзывы в train, а в validation и test сохраняйте исходный порядок.

[Документация DataCollatorWithPadding](https://huggingface.co/docs/transformers/main_classes/data_collator)

<details>
<summary><b>Нужна подсказка?</b></summary>

Передайте созданный `ReviewDataset` в `DataLoader`. За padding отвечает аргумент `collate_fn`, а перемешивание нужно только обучающей выборке.

</details>

In [ ]:
collator = DataCollatorWithPadding(bert_tokenizer, return_tensors="pt")

def make_loader(frame, batch_size, shuffle):
    # YOUR CODE HERE  ฅ^•ﻌ•^ฅ
    # Верните DataLoader с ReviewDataset и collator.
    pass


# Создайте train_loader, val_loader и test_loader.

In [ ]:
first_batch = next(iter(train_loader))
print({name: tuple(tensor.shape) for name, tensor in first_batch.items()})
assert first_batch["input_ids"].shape == first_batch["attention_mask"].shape

Теперь, когда батч готов, нужна модель, которая поставит каждому отзыву три оценки (`logits`), по одной на класс. Для этого поверх BERT добавим небольшой обучаемый слой. Его называют классификационной головой.

При загрузке модель может предупредить, что у этого слоя случайные веса. Слой создан для нашей задачи и ещё не обучен. Оптимизатор AdamW будет обновлять веса модели.

**Задача.** Загрузите модель на три класса и настройте AdamW с `lr=2e-5`.

<details>
<summary><b>Нужна подсказка?</b></summary>

Модель создаётся через `AutoModelForSequenceClassification.from_pretrained`; число классов и словари названий передаются при загрузке. Optimizer — `torch.optim.AdamW` по параметрам модели.

</details>

In [ ]:
id2label = {i: name for i, name in enumerate(LABEL_NAMES)}
label2id = {name: i for i, name in id2label.items()}

# YOUR CODE HERE  ฅ^•ﻌ•^ฅ
# Создайте model с тремя классами и optimizer AdamW с lr=2e-5.

После каждой эпохи будем проверять, стало ли лучше на validation. Для этого нужны оценки модели, правильные классы и средний loss — число, которое показывает, насколько оценки расходятся с правильными ответами. Из оценок потом получим accuracy и macro-F1.

**Задача.** Напишите `collect_predictions`. Обойдите батчи из переданного `loader` с текущими весами модели. Верните оценки модели (`logits`), правильные классы и средний loss на отзыв.

<details>
<summary><b>Нужна подсказка?</b></summary>

Для проверки нужны `model.eval()` и `torch.inference_mode()`. Сохраняйте logits и labels на CPU, а в конце объедините части через `torch.cat`. Средний loss считайте по объектам, а не по числу батчей.

</details>

In [ ]:
def collect_predictions(model, loader):
    model.eval()
    all_logits = []
    all_labels = []
    total_loss = 0.0
    total_objects = 0

    # YOUR CODE HERE  ฅ^•ﻌ•^ฅ
    # Верните logits, правильные ответы и средний loss по объектам.

    return torch.cat(all_logits), torch.cat(all_labels), total_loss / total_objects

In [ ]:
def metrics_from_logits(logits, labels, loss):
    predictions = logits.argmax(dim=1).numpy()
    probabilities = logits.softmax(dim=1).numpy()
    labels = labels.numpy()
    return {
        "loss": loss,
        "accuracy": accuracy_score(labels, predictions),
        "macro_f1": f1_score(labels, predictions, average="macro"),
        "predictions": predictions,
        "probabilities": probabilities,
        "targets": labels,
    }

Один проход по всем обучающим батчам называют эпохой. В каждом батче модель предсказывает классы, считает loss по известным ответам и обновляет веса.

**Задача.** Напишите `train_one_epoch`. Обучите модель на всех батчах из `loader` и верните средний loss на отзыв за эпоху.

<details>
<summary><b>Нужна подсказка?</b></summary>

В режиме обучения для каждого батча нужно очистить старые градиенты, получить loss, посчитать новые градиенты и обновить параметры. Вспомните методы `zero_grad`, `backward` и `step`.

</details>

In [ ]:
def train_one_epoch(model, loader, optimizer):
    model.train()
    total_loss = 0.0
    total_objects = 0

    # YOUR CODE HERE  ฅ^•ﻌ•^ฅ
    # Проведите одну эпоху и верните средний loss по объектам.

    return total_loss / total_objects

После двух эпох результат может быть хуже, чем после первой. Поэтому после каждой эпохи будем считать macro-F1 на validation и сохранять веса, которые дали лучший результат. `state_dict` хранит состояние модели, включая веса.

**Задача.** Запустите две эпохи. После каждой посчитайте macro-F1 на validation. В конце загрузите в модель сохранённое лучшее состояние.

<details>
<summary><b>Нужна подсказка?</b></summary>

После каждой эпохи используйте уже написанные функции обучения и проверки. При улучшении validation macro-F1 сохраните копию `state_dict`; здесь пригодится `copy.deepcopy`. В конце верните лучшее состояние через `load_state_dict`.

</details>

In [ ]:
EPOCHS = 2
best_macro_f1 = -1.0
best_state = None

# YOUR CODE HERE  ฅ^•ﻌ•^ฅ
# Обучите модель EPOCHS эпох и после каждой проверяйте её на validation.
# В конце в model должно остаться состояние с лучшим macro-F1.

## 4. Сравнение на test

На validation мы выбрали эпоху BERT. Теперь проверим три готовые модели на одной и той же отложенной выборке.

**Задача.** Получите предсказания на test от TF-IDF, замороженного BERT и дообученного BERT. Покажите в одной таблице их accuracy и macro-F1.

<details>
<summary><b>Нужна подсказка?</b></summary>

Для двух sklearn-моделей нужен `predict`, для BERT — `collect_predictions` и `metrics_from_logits`. Не вызывайте никакой `fit` на test.

</details>

In [ ]:
# YOUR CODE HERE  ฅ^•ﻌ•^ฅ
# Получите предсказания TF-IDF и замороженного BERT на test.
# Оцените дообученный BERT через collect_predictions и metrics_from_logits.
# Соберите final_results с accuracy и macro-F1 трёх способов.

Accuracy показывает, какая доля отзывов классифицирована правильно. Положительных отзывов больше всего, и они сильнее влияют на это число. Macro-F1 даёт каждому классу одинаковый вес.

В логистической регрессии мы включили `class_weight="balanced"`. При этом ошибки на редких классах получают больший вес. При дообучении BERT у каждого отзыва был одинаковый вес. Посмотрим, на каких классах это могло сказаться.

## 5. Анализ ошибок

В итоговой таблице по два числа на модель. Чтобы понять, какие классы путает дообученный BERT, построим матрицу ошибок. По строкам будут правильные классы, по столбцам — прогнозы модели. На диагонали окажутся правильные ответы.

**Задача.** Постройте эту матрицу и найдите десять ошибок, в которых модель была наиболее уверена. В коде уверенность — вероятность, которую модель присвоила выбранному классу. Посмотрите, куда чаще всего попали нейтральные отзывы.

<details>
<summary><b>Нужна подсказка?</b></summary>

Таблицу «правильный класс × предсказанный класс» можно получить через `pd.crosstab`. Уверенные ошибки — это неверные строки после сортировки по максимальной вероятности.

</details>

In [ ]:
analysis = test[["rating", "input", "label"]].copy()
analysis["fine-tuned BERT"] = finetuned_test["predictions"]
analysis["уверенность"] = finetuned_test["probabilities"].max(axis=1)

# YOUR CODE HERE  ฅ^•ﻌ•^ฅ
# Постройте матрицу ошибок.
# Выберите 10 неправильных ответов с наибольшей уверенностью.

### Проверим модель на своих отзывах

На test мы сравнили предсказания с классами, которые получили из звёзд. Теперь представьте, что пришёл новый отзыв и у нас есть только его текст. Какой класс предскажет модель, если автор хвалит товар, но ругает доставку?

**Задача.** Напишите `predict_texts`. На входе список отзывов, на выходе по одному номеру класса для каждого отзыва. Проверьте функцию на `custom_reviews`. Можно добавить свои примеры. Покажите таблицу с текстами и названиями предсказанных классов и сравните прогнозы со своей оценкой.

<details>
<summary><b>Нужна подсказка?</b></summary>

Токенизируйте список отзывов одним батчем и передайте его модели. Для предсказания пригодятся `model.eval()` и `torch.inference_mode()`, как при проверке на test. Для каждого текста модель выдаёт три оценки (`logits`), по одной на класс. `argmax` вернёт номер класса с самой высокой оценкой.

</details>

In [ ]:
custom_reviews = [
    "не могу не рекомендовать", "качество огонь, доставка ужас",
    "ну такое", "товар имба, закажу ещё", "оооочень довольна!!!",
]

def predict_texts(texts):
    # YOUR CODE HERE  ฅ^•ﻌ•^ฅ
    # Верните по одному номеру класса на отзыв.
    pass


# Получите custom_pred и покажите названия классов рядом с отзывами.

## Итог

В контрольном запуске дообученный BERT получил самую высокую accuracy. По macro-F1 первым оказался TF-IDF. Матрица ошибок показывает, что дообученный BERT плохо распознавал редкий нейтральный класс. В бонусе можно проверить, поможет ли больший вес для редких классов при обучении.

In [ ]:
SAVE_DIR = "rubert-reviews"
model.save_pretrained(SAVE_DIR)
bert_tokenizer.save_pretrained(SAVE_DIR)
print("Модель сохранена в", SAVE_DIR)

## Бонус для дома

1. Загрузите сохранённую модель и проверьте предсказания.
2. Посмотрите, какая часть длинного отзыва останется при обрезке до 64, 128 и 256 токенов. Это называется truncation.
3. Попробуйте дать ошибкам на редких классах больший вес при обучении. Сравните macro-F1 и матрицы ошибок на validation.

<details>
<summary><b>Нужна подсказка?</b></summary>

И токенизатор, и модель загружаются методом `from_pretrained` из одной папки. Совпадение двух массивов предсказаний можно проверить через `np.array_equal`.

</details>

In [ ]:
# YOUR CODE HERE  ฅ^•ﻌ•^ฅ
# Загрузите tokenizer и model из SAVE_DIR.
# Проверьте, что предсказания на custom_reviews совпали.

<details>
<summary><b>Нужна подсказка?</b></summary>

Получите `input_ids` с нужным `max_length`, а затем превратите их обратно в текст методом `decode`. Повторите это для одного и того же длинного отзыва.

</details>

In [ ]:
token_lengths = test["input"].map(
    lambda text: len(bert_tokenizer(text, truncation=False)["input_ids"])
)
long_text = test.loc[token_lengths.idxmax(), "input"]

# YOUR CODE HERE  ฅ^•ﻌ•^ฅ
# Для max_length 64, 128 и 256 выведите оставшуюся часть long_text.

Полезные ссылки:

- [Классификация текста в Transformers](https://huggingface.co/docs/transformers/tasks/sequence_classification)
- [Dataset и DataLoader](https://pytorch.org/docs/stable/data.html)
- [AdamW](https://pytorch.org/docs/stable/generated/torch.optim.AdamW.html)
- [DataCollatorWithPadding](https://huggingface.co/docs/transformers/main_classes/data_collator)
- [LogisticRegression](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html)